# Qwen3.8-Flash-Next NVFP4 on 2 × DGX Spark

## TL;DR

**Measured:** 47.54 output tok/s at concurrency 1, 275.37 aggregate output tok/s at concurrency 16, and 2,960.12 input tok/s for an uncached 16K prompt. This notebook preserves the clean measured output and is the runnable controller path for the exact two-node SGLang recipe.

![Measured performance](assets/performance.png)

Evidence class: **MEASURED**. Exact public recipe: [qwen3-8-flash-next-sglang-2x-dgx-spark](https://github.com/PixelML/qwen3-8-flash-next-sglang-2x-dgx-spark/tree/682504bec9e7e99206212f4e172b7ec823e4605c).

## Requirements

- Two DGX Spark systems on the same system-software release, one GB10 per node, with a working direct RoCE link.
- A Jupyter controller with passwordless SSH aliases to both nodes and Docker access on each node.
- At least 160 GB of local model storage on **each** node. This recipe never assumes a cross-cluster shared mount.
- A private local environment file based on the detailed recipe's `.env.example`; never commit it or print its addresses/devices.
- Safety stop: do not continue with a conflicting GPU workload, missing accelerator, storage error, OOM/Xid, or breached thermal limit.

## Configure

Set the values through local environment variables. The notebook stores no machine names, addresses, interface names, credentials, or private paths. Change `PIXELML_RUN_LIVE` to `1` only on the authorized two-node controller.

In [1]:
from pathlib import Path
import csv, json, os, subprocess, sys, time

RUN_LIVE = os.environ.get("PIXELML_RUN_LIVE", "0") == "1"
NODE_1 = os.environ.get("PIXELML_DGX_NODE_1", "")
NODE_2 = os.environ.get("PIXELML_DGX_NODE_2", "")
REMOTE_INSTALL_DIR = os.environ.get("PIXELML_DGX_INSTALL_DIR", "")
DGX_ENV_FILE = Path(os.environ.get("PIXELML_DGX_ENV_FILE", "/nonexistent"))
API_SECRET_FILE = Path(os.environ.get("PIXELML_API_SECRET_FILE", "/tmp/pixelml-qwen-dgx-api-key"))
API_BASE = os.environ.get("PIXELML_API_BASE", "")
PROMPT = "Explain why speculative decoding helps single-stream latency."

RECIPE_DIR = Path.cwd().resolve()
if not (RECIPE_DIR / "recipe.json").exists():
    RECIPE_DIR = (RECIPE_DIR / "recipes" / "qwen3.8-flash-next-sglang").resolve()
REPO_ROOT = RECIPE_DIR.parents[1]
PINS = json.loads((RECIPE_DIR / "recipe.json").read_text())
print(json.dumps({"run_live": RUN_LIVE, "topology": "2 x DGX Spark, TP=2", "pins": {"model_revision": PINS["model_revision"], "runtime_pin": PINS["runtime_pin"]}}, indent=2))

{
  "run_live": false,
  "topology": "2 x DGX Spark, TP=2",
  "pins": {
    "model_revision": "b80180e371f13348ec49641a6e66999e7854b179",
    "runtime_pin": "recipe 682504bec9e7...; pinned SGLang image digest"
  }
}

## Preflight

The live gate resolves both nodes at run time and prints only generic node labels plus safe GPU fields. It never prints SSH targets, network addresses, interface names, UUIDs, or process command lines.

In [2]:
def run(command, **kwargs):
    return subprocess.run(command, check=True, text=True, **kwargs)

if not RUN_LIVE:
    print("RECORDED MODE — clean measured outputs below; set local variables and PIXELML_RUN_LIVE=1 to reproduce.")
else:
    missing = [name for name, value in {"PIXELML_DGX_NODE_1": NODE_1, "PIXELML_DGX_NODE_2": NODE_2, "PIXELML_DGX_INSTALL_DIR": REMOTE_INSTALL_DIR, "PIXELML_API_BASE": API_BASE}.items() if not value]
    assert not missing, "missing local configuration: " + ", ".join(missing)
    assert DGX_ENV_FILE.is_file(), "PIXELML_DGX_ENV_FILE must point to a private local .env file"
    probe = "nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu,temperature.gpu,driver_version --format=csv,noheader,nounits"
    for label, node in (("node-1", NODE_1), ("node-2", NODE_2)):
        result = run(["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=8", node, probe], capture_output=True).stdout.strip()
        name, total, used, util, core, driver = [part.strip() for part in result.split(",")]
        assert float(total) >= 120000, f"{label}: expected GB10 memory class"
        assert float(used) < 2048 and float(util) < 5, f"{label}: GPU is not free"
        assert float(core) < 80, f"{label}: core temperature is unsafe"
        print(f"{label}: {name}, {total} MiB, {used} MiB used, {util}% util, {core} C, driver {driver}")
    print("PREFLIGHT PASS")

RECORDED MODE — clean measured outputs below; set local variables and PIXELML_RUN_LIVE=1 to reproduce.

## Install the pinned recipe on both nodes

This idempotent cell checks out the exact public recipe commit on the controller and both nodes. Private configuration and the API secret are copied only to the local working directories and remain ignored.

In [ ]:
RECIPE_SHA = "682504bec9e7e99206212f4e172b7ec823e4605c"
CONTROLLER_REPO = Path("/tmp/pixelml-qwen-dgx-recipe")
if RUN_LIVE:
    if not (CONTROLLER_REPO / ".git").exists():
        run(["git", "clone", "https://github.com/PixelML/qwen3-8-flash-next-sglang-2x-dgx-spark", str(CONTROLLER_REPO)])
    run(["git", "-C", str(CONTROLLER_REPO), "checkout", "--detach", RECIPE_SHA])
    if not API_SECRET_FILE.exists():
        API_SECRET_FILE.write_text(subprocess.check_output(["openssl", "rand", "-hex", "32"], text=True).strip())
        API_SECRET_FILE.chmod(0o600)
    for node in (NODE_1, NODE_2):
        remote = f'''set -euo pipefail
if [ ! -d "{REMOTE_INSTALL_DIR}/.git" ]; then git clone https://github.com/PixelML/qwen3-8-flash-next-sglang-2x-dgx-spark "{REMOTE_INSTALL_DIR}"; fi
git -C "{REMOTE_INSTALL_DIR}" fetch origin {RECIPE_SHA}
git -C "{REMOTE_INSTALL_DIR}" checkout --detach {RECIPE_SHA}
'''
        run(["ssh", node, "bash", "-s"], input=remote)
        run(["scp", str(DGX_ENV_FILE), f"{node}:{REMOTE_INSTALL_DIR}/.env"])
        run(["scp", str(API_SECRET_FILE), f"{node}:{REMOTE_INSTALL_DIR}/.sglang-api-key"])
    run(["cp", str(DGX_ENV_FILE), str(CONTROLLER_REPO / ".env")])
    run(["cp", str(API_SECRET_FILE), str(CONTROLLER_REPO / ".sglang-api-key")])
    print("PINNED RECIPE READY ON BOTH NODES")
else:
    print("Install skipped in recorded mode.")

## Prepare model storage on each node

The exact checkpoint revision is downloaded and verified separately on each node's configured local storage. Both downloads run in parallel; neither node depends on a shared volume.

In [ ]:
if RUN_LIVE:
    processes = []
    for node in (NODE_1, NODE_2):
        command = f'cd "{REMOTE_INSTALL_DIR}" && ./scripts/prepare-model.sh'
        processes.append(subprocess.Popen(["ssh", node, command]))
    failures = [process.wait() for process in processes]
    assert failures == [0, 0], f"model preparation failed: {failures}"
    print("MODEL INTEGRITY PASS ON BOTH NODES")
else:
    print("Model preparation skipped in recorded mode.")

## Start the two-node service

The pinned lifecycle script launches both ranks, applies the SM121 compatibility patches, waits for health, and exposes the OpenAI-compatible API on rank zero.

In [ ]:
if RUN_LIVE:
    run(["bash", "scripts/start-cluster.sh"], cwd=CONTROLLER_REPO)
    health = run(["curl", "-fsS", "--max-time", "8", API_BASE.rstrip("/").replace("/v1", "") + "/health"], capture_output=True)
    print("SERVICE READY" if health.returncode == 0 else "SERVICE NOT READY")
else:
    print("Service start skipped in recorded mode.")

## Benchmark and recorded output

The live benchmark uses the detailed repository's functional, concurrency, and unique-prefix prefill harnesses. Token counts come from the final API usage object; the notebook refuses event-counted estimates.

In [3]:
if RUN_LIVE:
    environment = dict(os.environ, PIXELML_API_BASE=API_BASE)
    run([sys.executable, "scripts/smoke-benchmark.py", "--base-url", API_BASE, "--secret-file", str(API_SECRET_FILE)], cwd=CONTROLLER_REPO, env=environment)
    run([sys.executable, "scripts/prefill-benchmark.py", "--base-url", API_BASE, "--secret-file", str(API_SECRET_FILE)], cwd=CONTROLLER_REPO, env=environment)

rows = list(csv.DictReader((RECIPE_DIR / "results/summary.csv").open()))
print("metric | x | completion_tokens | wall_or_ttft_seconds | tok_s")
print("--- | ---: | ---: | ---: | ---:")
for row in rows:
    print(" | ".join(row[column] for column in ("metric", "x", "completion_tokens", "wall_or_ttft_seconds", "tok_s")))

metric | x | completion_tokens | wall_or_ttft_seconds | tok_s
--- | ---: | ---: | ---: | ---:
decode | 1 | 192 | 4.038 | 47.54
decode | 4 | 750 | 8.567 | 87.55
decode | 8 | 1536 | 9.711 | 158.17
decode | 16 | 3072 | 11.156 | 275.37
prefill | 1046 | 1 | 0.4524 | 2327.78
prefill | 4103 | 1 | 1.4924 | 2758.65
prefill | 16471 | 1 | 5.5729 | 2960.12

In [4]:
if RUN_LIVE:
    try:
        import PIL  # noqa: F401
    except ImportError:
        run([sys.executable, "-m", "pip", "install", "Pillow"])
run([
    sys.executable, str(REPO_ROOT / "scripts/render_recipe_chart.py"),
    "--spec", str(RECIPE_DIR / "chart-spec.json"),
    "--output", str(RECIPE_DIR / "assets/performance.png"),
])
print("Chart regenerated from committed results: assets/performance.png")

Chart regenerated from committed results: assets/performance.png

## Try your own prompt

Edit `PROMPT` below. With the service ready, the final `curl` prints the answer and the authoritative `prompt_tokens`, `completion_tokens`, and `total_tokens` fields.

In [ ]:
PROMPT = "Write a compact Python function that validates a topological ordering. Return code only."
os.environ["PIXELML_PROMPT"] = PROMPT
os.environ["PIXELML_API_BASE"] = API_BASE
os.environ["PIXELML_API_SECRET_FILE"] = str(API_SECRET_FILE)
print(PROMPT)

In [ ]:
%%bash
set -euo pipefail
if [ "${PIXELML_RUN_LIVE:-0}" != "1" ]; then
  echo "Set local configuration and PIXELML_RUN_LIVE=1, rerun from Configure, then run this cell."
  exit 0
fi
BODY=$(jq -n --arg prompt "$PIXELML_PROMPT" '{model:"qwen3.8-flash-next",messages:[{role:"user",content:$prompt}],max_tokens:256,temperature:0.0,reasoning_effort:"low",stream:false}')
RESPONSE=$(curl -sS "$PIXELML_API_BASE/chat/completions" \
  -H "Authorization: Bearer $(<"$PIXELML_API_SECRET_FILE")" \
  -H 'Content-Type: application/json' \
  -d "$BODY")
printf '%s\n' "$RESPONSE" | jq -r '.choices[0].message.content'
printf '\nusage:\n'
printf '%s\n' "$RESPONSE" | jq '.usage | {prompt_tokens, completion_tokens, total_tokens}'